In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## **Análisis de las estrellas de nuestra galaxia en un campo de observación**

El pripeline hace lo siguiente:

`ADQL` contiene el query asociado a nuestra consulta, para ello pedimos las coordenadas asociadas a GAIA y las magnitudes de las tablas de GAIA y WISE. Para diferenciar entonces las tablas definimos a GAIA con la letra g y a WISE con w.

Luego le pedimos hacer el INNER JOIN entre la tabla de GAIA, que tomamos como principal, con la tabla de WISE (secundaria en este caso).

Para las condiciones le decimos que en un círculo de 0.5 grados busque en la región de coordenadas (135.5, 0.5) (asención recta y declinación) las estrellas que estén allí y coincidan además. Incluso que relacione la distancia con el comando `DISTANCE` en 2 arcosegundos.



In [ ]:
%%bash

echo "Pipeline de consulta en VizieR"

ADQL="
SELECT
g.RA_ICRS,
g.DE_ICRS,
g.Gmag,
w.W1mag
FROM \"I/355/gaiadr3\" AS g
INNER JOIN \"II/328/allwise\" AS w
  ON 1=CONTAINS(
    POINT('ICRS', g.RA_ICRS, g.DE_ICRS),
    CIRCLE('ICRS', 135.5, 0.5, 0.5))
  AND 1=CONTAINS(
    POINT('ICRS', w.RAJ2000, W.DEJ2000),
    CIRCLE('ICRS', 135.5, 0.5, 0.5)
  )
  AND DISTANCE(
    POINT('ICRS', g.RA_ICRS, g.DE_ICRS),
    POINT('ICRS', w.RAJ2000, w.DEJ2000)
  ) < 2.0/3600.0
"

URL_ADQL=$(echo $ADQL | sed 's/ /+/g')

TAP_URL="https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?request=doQuery&lang=ADQL&format=csv&query="

echo $TAP_URL$URL_ADQL

wget -q -O mw_stars.csv "$TAP_URL$URL_ADQL"

## **Lectura de datos**

Aquí vamos a leer primero los datos que tenemos de neustra consulta que quedaron guardados en nuestro archivo `mw_stars.csv`.

Los datos solicitados fueron la `Gmag`y la `W1mag`, y tienen para cada estrella las coordenadas de Asención Recta y Declinación respectivas con la referencia de J2000.

In [ ]:
df_stars_mw = pd.read_csv('mw_stars.csv').dropna()

df_stars_mw.head(10)

In [ ]:
print(len(df_stars_mw))

## **Diagrama color-magnitud de las estrellas en la Vía Láctea**

Para nuestro diagrama color magnitud calcularemos el índice de color entre el visible y el infrarrojo:

$$
C = V - I
$$

Donde $C$ representa el índice de color dado por la resta entre las magnitudes de las dos tablas.

In [ ]:
df_stars_mw.keys()

Por simplicidad vamos a ver cuántas estrellas hay por cada magnitud, en un rango de magnitudes claro. Para esto visualizaremos un histograma:

In [ ]:
#@title Cuenta de estrellas en las dos magnitudes
plt.style.use('dark_background')

cuentas, axs = plt.subplots(1,2, figsize=(10,5))

axs[0].hist(df_stars_mw['Gmag'], bins=50, color='green')
axs[0].set_title("Estrellas en magnitud visible (GAIA)")
axs[0].set_xlabel('Magnitud G')
axs[0].set_ylabel('Número de estrellas')
axs[0].grid(True)

axs[1].hist(df_stars_mw['W1mag'], bins=50, color='red')
axs[1].set_title("Estrellas en magnitud infrarroja (WISE)")
axs[1].set_xlabel('Magnitud I')
axs[1].grid(True)

Ahora calcularemos el índice de color y lo añadiremos al DataFrame como una nueva columna:

In [ ]:
df_stars_mw["color_index"] = df_stars_mw['Gmag'] - df_stars_mw['W1mag']

Crearemos entonces nuestro diagrama color-magnitud para las estrellas de la Vía Láctea en esta región:

In [ ]:
plt.style.use('dark_background')

plt.figure(figsize=(8,8))
plt.scatter(df_stars_mw['Gmag'], df_stars_mw['color_index'])
plt.gca().invert_xaxis()